# 03 — Final Benchmark (the spine)

This is where the three experiment phases come together into **one comparable result**.

It takes a short list of **finalists** — your best prompt(s) from
`02_prompt_variance` combined with a `reasoning_effort` level from `01_model_selection` —
runs them all on the **same model (GPT-5)** so the numbers are apples-to-apples, then
for each finalist reports:

| Axis | Why it matters |
|------|----------------|
| **AUC** | threshold-free ranking quality |
| **F1 (default)** | minority-class F1 at the model's raw 0/1 output |
| **F1 (tuned)** | F1 after moving the decision threshold to its best point (tuned on the tuning sample, frozen, applied out-of-sample — no leakage) |
| **Cost / 100 decisions** | $ spent per 100 loans scored — the axis Sabadell cares about |

…all next to **XGBoost** (also threshold-tuned) as the classical-ML baseline.

> **This notebook spends real API money** (it calls GPT-5). Nothing runs until you
> fill in the `FINALISTS` list and run the cells. Tuning/threshold cells that don't
> hit the API are free to re-run.

## Setup

In [ ]:
# llm_utils.py / llm_pricing.py live two directories up (llm_models/).
import sys; sys.path.insert(0, '..')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)

from llm_utils import (
    load_llm_sample, run_ml_on_sample, run_llm_experiment,
    find_best_threshold, build_system_prompt, build_user_prompt,
    DATA_DIR, RESULTS_DIR, LLM_CALLS_PATH,
)

load_dotenv('../.env', override=False)
MODEL = 'gpt-5.4'          # the chosen model from 01_model_selection
API_KEY = os.environ.get('OPENAI_API_KEY')
assert API_KEY, 'Set OPENAI_API_KEY in notebooks/llm_models/.env'

## Step 1 — Define your finalists

Pick the **2–3 best prompts** from `02_prompt_variance` and pair each with a
`reasoning_effort` level. Each finalist needs:

- `name` — a short unique label (used to tag cost rows in `llm_calls.csv`)
- `system_prompt` — the variant's system prompt (`None` = the default `build_system_prompt()`)
- `user_prompt_fn` — `None` = default `build_user_prompt`; or a `lambda row: ...` for variants
  like `top_features_only`
- `reasoning_effort` — `'low' | 'medium' | 'high'`

> **TODO (you):** copy the winning prompt definitions out of
> `02_prompt_variance/02_Prompt_Variance.ipynb` (the `PROMPT_VARIANTS` list).
> Tip for later: lifting `PROMPT_VARIANTS` into a shared `prompt_variants.py` would let
> both notebooks import the same definitions instead of copy-pasting.

In [ ]:
FINALISTS = [
    # --- EDIT ME: 2-3 finalists ---
    dict(name='baseline-medium',
         system_prompt=None,            # None -> build_system_prompt()
         user_prompt_fn=None,           # None -> build_user_prompt(row, include_desc=False)
         reasoning_effort='medium'),
    # dict(name='top_features-high',
    #      system_prompt=None,
    #      user_prompt_fn=lambda row: build_user_prompt(row, include_desc=False),
    #      reasoning_effort='high'),
]
assert 1 <= len(FINALISTS) <= 4, 'Keep it to a handful — this costs money.'
print('Finalists:', [f['name'] for f in FINALISTS])

## Step 2 — Run finalists on the TUNING sample (GPT-5)

Runs each finalist on the 100-loan tuning sample (`tuning_sample.csv`). This is where
we learn each finalist's best decision threshold. Cost is logged automatically to
`llm_calls.csv` under each finalist's `name`.

In [ ]:
tune_sample = load_llm_sample()
y_tune = tune_sample['loan_status'].values

tune_runs = {}
for f in FINALISTS:
    print(f"\n=== TUNING RUN: {f['name']} ===")
    tune_runs[f['name']] = run_llm_experiment(
        tune_sample, api_provider='openai', model_name=MODEL, api_key=API_KEY,
        label=f['name'], include_desc=False, with_logprobs=True,
        system_prompt=f['system_prompt'], user_prompt_fn=f['user_prompt_fn'],
        reasoning_effort=f['reasoning_effort'],
    )

## Step 3 — Tune the decision threshold per finalist

No API calls here — free to re-run. For each finalist we compare F1 at the raw 0/1
output vs F1 at the threshold that maximises minority-class (Charged Off) F1.

In [ ]:
tune_rows = []
for f in FINALISTS:
    r = tune_runs[f['name']]
    probs = np.array([p if p is not None else np.nan for p in r['probabilities']])
    preds = np.array(r['predictions'])
    m = ~pd.isna(probs)
    yt, pv, dv = y_tune[m], probs[m], preds[m]
    t, p_co, r_co, f1_tuned = find_best_threshold(yt, pv)
    tune_rows.append(dict(
        finalist=f['name'], reasoning_effort=f['reasoning_effort'], n=int(m.sum()),
        auc=roc_auc_score(yt, pv) if len(set(yt)) == 2 else np.nan,
        f1_default=f1_score(yt, dv, pos_label=0, zero_division=0),
        f1_tuned=f1_tuned, threshold_tuned=t,
    ))
tune_table = pd.DataFrame(tune_rows).set_index('finalist')
print(tune_table.to_string(float_format=lambda x: f'{x:.4f}'))

## Step 4 — Held-out test (the honest numbers)

Each finalist's **frozen** threshold (from Step 3) is applied to a *different* set of
loans — the held-out batch `test_batch.csv` — so nothing is tuned on the data
it's scored on. XGBoost is run on the same batch as the classical baseline.

> Regenerate via `sample_generation.get_test_batch(force=True)` if the test file
> predates the 30-feature preprocessing.

In [ ]:
held = pd.read_csv(f'{DATA_DIR}/test_batch.csv')
y_held = held['loan_status'].values
print(f'Held-out: {len(held)} loans | Charged Off {(y_held==0).sum()} | Fully Paid {(y_held==1).sum()}')

# Classical baseline
xgb_probs, xgb_preds = run_ml_on_sample(held)
xgb_t, xgb_p, xgb_r, xgb_f1 = find_best_threshold(y_held, xgb_probs)
xgb_preds_tuned = (xgb_probs >= xgb_t).astype(int)

results = []
for f in FINALISTS:
    print(f"\n=== HELD-OUT RUN: {f['name']} ===")
    hr = run_llm_experiment(
        held, api_provider='openai', model_name=MODEL, api_key=API_KEY,
        label=f"{f['name']} (held-out)", include_desc=False, with_logprobs=True,
        system_prompt=f['system_prompt'], user_prompt_fn=f['user_prompt_fn'],
        reasoning_effort=f['reasoning_effort'],
    )
    probs = np.array([p if p is not None else np.nan for p in hr['probabilities']])
    preds_def = np.array(hr['predictions'])
    m = ~pd.isna(probs)
    yt, pv = y_held[m], probs[m]
    thr = float(tune_table.loc[f['name'], 'threshold_tuned'])
    preds_tuned = (pv >= thr).astype(int)
    results.append(dict(
        finalist=f['name'],
        accuracy=accuracy_score(yt, preds_tuned),
        auc=roc_auc_score(yt, pv) if len(set(yt)) == 2 else np.nan,
        f1_default=f1_score(yt, preds_def[m], pos_label=0, zero_division=0),
        f1_tuned=f1_score(yt, preds_tuned, pos_label=0, zero_division=0),
        recall_co=recall_score(yt, preds_tuned, pos_label=0, zero_division=0),
        precision_co=precision_score(yt, preds_tuned, pos_label=0, zero_division=0),
        threshold=thr,
    ))

## Step 5 — Add the cost axis

Cost was logged per call to `llm_calls.csv` during every run above. We sum it per
finalist and express it as **$ per 100 loans scored** — the unit that lets you say:
*"this prompt buys +X F1 but costs Y× more per decision."*

In [ ]:
calls = pd.read_csv(LLM_CALLS_PATH)
# Cost across both the tuning and held-out runs, per finalist name.
def cost_per_100(name):
    sub = calls[calls['label'].str.startswith(name)]
    if sub.empty:
        return np.nan
    return 100.0 * sub['cost_usd'].sum() / sub['row_index'].count()

summary = pd.DataFrame(results).set_index('finalist')
summary['cost_per_100_usd'] = [cost_per_100(n) for n in summary.index]

# Append the XGBoost baseline (cost ~ $0 — local model).
summary.loc['XGBoost (baseline)'] = dict(
    accuracy=accuracy_score(y_held, xgb_preds_tuned), auc=roc_auc_score(y_held, xgb_probs),
    f1_default=f1_score(y_held, xgb_preds, pos_label=0, zero_division=0),
    f1_tuned=xgb_f1, recall_co=xgb_r, precision_co=xgb_p, threshold=xgb_t,
    cost_per_100_usd=0.0,
)
print(summary.to_string(float_format=lambda x: f'{x:.4f}'))

## Step 6 — The punchline: metric vs cost

The slide. Each finalist is a point; XGBoost sits on the $0 axis. Up-and-to-the-left
is better (more F1, less money). Whatever's on the upper-left frontier is your
recommendation.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, row in summary.iterrows():
    ax.scatter(row['cost_per_100_usd'], row['f1_tuned'], s=90)
    ax.annotate(name, (row['cost_per_100_usd'], row['f1_tuned']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.set_xlabel('Cost per 100 loans scored ($)')
ax.set_ylabel('F1 — Charged Off (tuned threshold)')
ax.set_title('Final benchmark: minority-class F1 vs cost (held-out)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/03_benchmark_f1_vs_cost.png', dpi=150)
plt.show()

summary.to_csv(f'{RESULTS_DIR}/03_final_benchmark.csv')
print(f'Saved 03_final_benchmark.csv + 03_benchmark_f1_vs_cost.png to {RESULTS_DIR}')

## How to read this for the presentation

- **AUC** is your fairest headline (threshold-free). Compare GPT-5 finalists vs XGBoost (~0.705).
- **F1 tuned vs default** shows how much the *decision rule* alone buys you — a cheap win
  that applies to both ML and LLM.
- **Cost per 100** turns the LLM's accuracy edge into a business trade-off: is +X F1 worth
  the per-decision price for a bank scoring thousands of loans? That framing is the
  capstone's strongest discussion point.